# Reviewing the ground-truth labels

The state-extraction methods (keyword gazetteer, Stanford NER, Gemma) are scored against `data/llm_pilot/labels.csv`: one reference answer for each of the 200 sampled posts. Claude wrote those labels. Before the scores mean anything, a person has to check them.

This notebook does that check in three parts:

1. **Look at the file** and see why it is awkward to review as it is (cells 1–2).
2. **Make a spreadsheet you can read**, edit it by hand (cells 3–5, then Excel).
3. **Read your edits back**, check them, save them, and re-score the methods (cells 6–9).

Run the cells in order. Nothing changes `labels.csv` until cell 7, and cell 7 keeps a backup first.

Full write-up: `docs/Label_Review.docx`.

## 1. Load the labels

`labels.csv` holds one row per post: Claude's answer (`human_states`, `human_main`, `human_event`), a confidence, a one-line reason (`note`), and the post text.

Every column is read as **text** (`dtype=str`). This matters for `post_id`: it is a 15–16 digit number, and read as a number it would be rounded and no longer match the database.

**Check:** the output should be `(200, 12)` — 200 posts, 12 columns.

In [1]:
# Run from a notebook in the repo's notebooks/ folder. Everything is read as text
# (dtype=str), so the 16-digit post IDs are never turned into rounded numbers.
from pathlib import Path
import re
import shutil
from datetime import date

import pandas as pd

FOLDER = Path("../data/llm_pilot")       # where labels.csv lives
labels = pd.read_csv(FOLDER / "labels.csv", dtype=str, keep_default_na=False)
labels.shape

(200, 12)

## 2. Examine the file

Before changing anything, see what is in it. This cell counts three things that make the CSV hard to review in Excel:

- **Line breaks inside posts.** Excel shows such a post across many lines, so it is hard to see where one post ends and the next begins.
- **Posts longer than 32,767 characters**, the most an Excel cell can hold. Excel cuts them without saying so.
- And a summary of the labels themselves: how many posts report a US event, how many name a state, how many Claude was unsure about (`confidence = low`).

**Check:** 100 English and 100 Spanish posts; 153 with line breaks; 5 too long for a cell; 40 marked low confidence.

In [36]:
labels.post_text.str.contains("\n").sum()

np.int64(153)

In [2]:
print("columns:", list(labels.columns))
print("posts by language:", labels.lang.value_counts().to_dict())

length = labels.post_text.str.len()
print("posts with line breaks:", labels.post_text.str.contains("\n").sum())
print("posts longer than an Excel cell (32,767 characters):", (length > 32_767).sum())
print("longest post:", length.max(), "characters")

print("\nposts reporting a US event:", (labels.human_event == "1").sum())
print("posts with at least one state:", (labels.human_states != "").sum())
print("labels marked low confidence:", (labels.confidence == "low").sum())

labels[["row", "lang", "human_states", "human_main", "human_event", "confidence", "note"]].head(10)

columns: ['row', 'post_id', 'lang', 'geo_tier', 'geo_matched', 'human_states', 'human_main', 'human_event', 'confidence', 'note', 'labelled_by', 'post_text']
posts by language: {'en': 100, 'es': 100}
posts with line breaks: 153
posts longer than an Excel cell (32,767 characters): 5
longest post: 49973 characters

posts reporting a US event: 140
posts with at least one state: 138
labels marked low confidence: 40


,row,lang,human_states,human_main,human_event,confidence,note
0,1,en,CA,CA,1,low,Main story is South Africa/Namibia; CA counted...
1,2,en,"KS,TX",TX,1,high,Dairy cows positive in Texas and Kansas; equal...
2,3,en,OR,OR,1,low,Oregon Zoo testing condor vaccine: counted as ...
3,4,en,TX,TX,1,low,Texas human case (April 2024) stated as curren...
4,5,en,LA,LA,1,high,Louisiana death is the US event. Southern Oreg...
5,6,en,"OR,TX",OR,1,high,Oregon house cat died from raw pet food (main ...
6,7,en,"AZ,LA,NV",NV,1,high,"Nevada new strain in dairy worker, Louisiana d..."
7,8,en,"NC,SC",NC,1,low,Detections in NC and SC wild waterfowl. Virgin...
8,9,en,MS,MS,1,high,H7N9 detected in Mississippi. Tennessee 2017 o...
9,10,en,TX,TX,1,low,Mostly airborne-spread research. Texas dairy-w...


In [37]:
labels.head(10)

,row,post_id,lang,geo_tier,geo_matched,human_states,human_main,human_event,confidence,note,labelled_by,post_text
0,1,1165978712756899,en,T5_multi,"CA,GA",CA,CA,1,low,Main story is South Africa/Namibia; CA counted...,"claude-opus-5-5, 2026-11-03, not yet reviewed",Bird Flu Could Pose New Threat to Namibia's Ca...
1,2,2024682845035614,en,T5_multi,"KS,TX","KS,TX",TX,1,high,Dairy cows positive in Texas and Kansas; equal...,"claude-opus-5-5, 2026-11-03, not yet reviewed",Unpasteurized milk samples and nose and throat...
2,3,328478749868262,en,T5_multi,"CA,OR",OR,OR,1,low,Oregon Zoo testing condor vaccine: counted as ...,"claude-opus-5-5, 2026-11-03, not yet reviewed",The Oregon Zoo is one of three zoos helping to...
3,4,1319321399090032,en,T5_multi,"CO,TX",TX,TX,1,low,Texas human case (April 2024) stated as curren...,"claude-opus-5-5, 2026-11-03, not yet reviewed","""A stockpiling operation that began 17 years a..."
4,5,1002127958174702,en,T5_multi,"LA,OR",LA,LA,1,high,Louisiana death is the US event. Southern Oreg...,"claude-opus-5-5, 2026-11-03, not yet reviewed",This could have far-reaching impacts on life a...
5,6,806732798284013,en,T5_multi,"CA,OR,TX","OR,TX",OR,1,high,Oregon house cat died from raw pet food (main ...,"claude-opus-5-5, 2026-11-03, not yet reviewed","""The risks of raw diets have become even more ..."
6,7,968374465466564,en,T5_multi,"AZ,LA,NV","AZ,LA,NV",NV,1,high,"Nevada new strain in dairy worker, Louisiana d...","claude-opus-5-5, 2026-11-03, not yet reviewed",Ongoing outbreaks of H5N1 avian influenza in t...
7,8,816572078033105,en,T5_multi,"NC,SC,VA","NC,SC",NC,1,low,Detections in NC and SC wild waterfowl. Virgin...,"claude-opus-5-5, 2026-11-03, not yet reviewed",U.S. Department of Agriculture has confirmed A...
8,9,1297088895553664,en,T5_multi,"MS,TN",MS,MS,1,high,H7N9 detected in Mississippi. Tennessee 2017 o...,"claude-opus-5-5, 2026-11-03, not yet reviewed",HPAI H7N9 Avian Influenza Detected in Mississi...
9,10,1116877900184471,en,T5_multi,"MD,TX",TX,TX,1,low,Mostly airborne-spread research. Texas dairy-w...,"claude-opus-5-5, 2026-11-03, not yet reviewed",Could the Bird Flu Become Airborne?\n\nScienti...


## 3. Put each post on one line

This makes a *display copy* of the text: line breaks become ` / `, repeated spaces collapse to one, and the five longest posts are cut to 32,000 characters with a note saying where the full text is.

It changes nothing in `labels.csv`. The original text stays there and in `sample.csv`.

**Check:** `line breaks left: 0`, and the longest cell is under 32,767.

In [39]:
EXCEL_MAX = 32_000          # a cell holds 32,767; leave room for the note below

def one_line(text):
    """Line breaks become ' / ' and repeated spaces become one."""
    t = re.sub(r"\s*[\r\n]+\s*", " / ", text or "")
    t = re.sub(r"[ \t]+", " ", t).strip()
    if len(t) > EXCEL_MAX:
        t = t[:EXCEL_MAX] + f" [cut here: full post is {len(text):,} characters, see sample.csv]"
    return t

shown = labels.post_text.map(one_line)
print("line breaks left:", shown.str.contains("\n").sum(), "| longest cell:", shown.str.len().max())
shown.head(3).tolist()

line breaks left: 0 | longest cell: 32059


['Bird Flu Could Pose New Threat to Namibia\'s Cape Fur Seals / Cape Fur Seals along Namibia\'s coastline may be facing a new and serious threat following the first confirmed case of highly pathogenic avian influenza, commonly known as bird flu, in a seal in South Africa. / Damian Schreiber, Marine Conservation Communications, Field and Research Officer at Ocean Conservation Namibia (OCN), said scientists are closely monitoring the situation after a rescued Cape Fur Seal in Mossel Bay, South Africa, tested positive for bird flu in April this year. "The seal was in very poor condition and had to be euthanised. Tests later confirmed it was infected with highly pathogenic avian influenza. This is the first recorded case of bird flu in a Cape Fur Seal," Schreiber explained. / He said the discovery is concerning because Cape Fur Seals are already dealing with rabies outbreaks and a range of human-related threats. According to Schreiber, there is a high risk that the disease could spread amo

## 4. Build the review table

One row per post with:

- `row` and `post_id` — which post. `post_id` is the key that links the row to the database.
- `claude_*`, `confidence`, `claude_note` — Claude's answer and reasoning, **for comparison only**.
- `final_states`, `final_main`, `final_event` — **your** answer. They start as copies of Claude's, so you only edit the rows you disagree with.
- `reviewed`, `comment` — mark `Y` once you have checked a row; add any note worth keeping.

In [41]:
# Claude's labels stay for comparison; the final_* columns start as copies, so only
# rows you disagree with need changing.
review = pd.DataFrame({
    "row": labels.row.astype(int),
    "post_id": labels.post_id,          # text, so all 16 digits survive; key for joining to the database
    "lang": labels.lang,
    "post_text": shown,
    "claude_states": labels.human_states,
    "claude_main": labels.human_main,
    "claude_event": labels.human_event,
    "confidence": labels.confidence,
    "claude_note": labels.note,
    "final_states": labels.human_states,
    "final_main": labels.human_main,
    "final_event": labels.human_event,
    "reviewed": "",
    "comment": "",
})
review.head()

,row,post_id,lang,post_text,claude_states,claude_main,claude_event,confidence,claude_note,final_states,final_main,final_event,reviewed,comment
0,1,1165978712756899,en,Bird Flu Could Pose New Threat to Namibia's Ca...,CA,CA,1,low,Main story is South Africa/Namibia; CA counted...,CA,CA,1,,
1,2,2024682845035614,en,Unpasteurized milk samples and nose and throat...,"KS,TX",TX,1,high,Dairy cows positive in Texas and Kansas; equal...,"KS,TX",TX,1,,
2,3,328478749868262,en,The Oregon Zoo is one of three zoos helping to...,OR,OR,1,low,Oregon Zoo testing condor vaccine: counted as ...,OR,OR,1,,
3,4,1319321399090032,en,"""A stockpiling operation that began 17 years a...",TX,TX,1,low,Texas human case (April 2024) stated as curren...,TX,TX,1,,
4,5,1002127958174702,en,This could have far-reaching impacts on life a...,LA,LA,1,high,Louisiana death is the US event. Southern Oreg...,LA,LA,1,,


## 5. Write the spreadsheet

Writes `data/llm_pilot/labels_review.xlsx`, formatted for reading:

- **Blue columns** are the ones you edit; leave the rest alone.
- **Yellow rows** are the 40 Claude marked low confidence — borderline calls to check first.
- `final_event` and `reviewed` have drop-down lists, so typos are unlikely.
- `post_id` is stored as **text**, so Excel keeps every digit.
- The header and post text stay in view while you scroll; rows are 90 points tall — drag one taller to read a long post.

### Now review in Excel

Open the file and, for each row you check:

- `final_states`: every US state with a bird-flu **event**, codes separated by commas (`KS,TX`), blank if none.
- `final_main`: the one state the post is mainly about; it must be one of `final_states`.
- `final_event`: `1` if the post reports a specific bird-flu event inside the US, `0` if not.
- `reviewed`: `Y`.

The rules are the ones the LLM prompt uses (Geolocation_Extraction.docx, section 6). An event is a detection, case, outbreak, cull, quarantine, or a state acting on an event in its own territory. A state is **not** counted when it is only part of a name (Lake Michigan, California condor), when "Washington" means the federal government, or when it appears only as history, a dateline, a link, a company headquarters or a price.

**Save and close the file before running cell 6.**

In [42]:
review.to_excel(FOLDER / "labels_review_basic.xlsx", index=False)

In [5]:
from openpyxl.formatting.rule import FormulaRule
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

out_path = FOLDER / "labels_review.xlsx"
widths = {"row": 6, "post_id": 19, "lang": 6, "post_text": 90, "claude_states": 14, "claude_main": 10,
          "claude_event": 10, "confidence": 10, "claude_note": 45, "final_states": 14,
          "final_main": 10, "final_event": 10, "reviewed": 9, "comment": 35}
editable = ["final_states", "final_main", "final_event", "reviewed", "comment"]

with pd.ExcelWriter(out_path, engine="openpyxl") as xw:
    review.to_excel(xw, sheet_name="labels", index=False)
    ws = xw.sheets["labels"]
    n = len(review) + 1                                   # last row, counting the header
    col = {name: get_column_letter(i) for i, name in enumerate(review.columns, start=1)}

    for i, name in enumerate(review.columns, start=1):
        letter = ws.cell(row=1, column=i).column_letter
        ws.column_dimensions[letter].width = widths[name]
        ws.cell(row=1, column=i).font = Font(bold=True)
        ws.cell(row=1, column=i).alignment = Alignment(wrap_text=True)
        for cell in ws[letter][1:]:
            cell.alignment = Alignment(vertical="top",
                                       wrap_text=name in ("post_text", "claude_note", "comment"))
            if name in editable:                          # blue = columns you edit
                cell.fill = PatternFill("solid", fgColor="EAF2FB")
            if name == "post_id":
                cell.number_format = "@"                  # Excel text format: never rounded
    for r in range(2, n + 1):
        ws.row_dimensions[r].height = 90                  # drag taller to read a long post

    ws.freeze_panes = f"{col['claude_states']}2"          # header and post text stay in view
    ws.auto_filter.ref = f"A1:{col['comment']}{n}"

    event_list = DataValidation(type="list", formula1='"1,0"')
    reviewed_list = DataValidation(type="list", formula1='"Y"', allow_blank=True)
    ws.add_data_validation(event_list); event_list.add(f"{col['final_event']}2:{col['final_event']}{n}")
    ws.add_data_validation(reviewed_list); reviewed_list.add(f"{col['reviewed']}2:{col['reviewed']}{n}")

    # low-confidence rows in yellow, to check first
    ws.conditional_formatting.add(f"A2:{col['claude_note']}{n}", FormulaRule(
        formula=[f'${col["confidence"]}2="low"'], fill=PatternFill("solid", fgColor="FFF4CC")))

print("wrote", out_path)

wrote ../data/llm_pilot/labels_review.xlsx


## 6. Read your review back and check it

Reads the saved spreadsheet and checks every row:

- every code is a real state code (lower case and spaces are fine; they are tidied);
- `final_main` is one of `final_states`, and a row with states has a main one;
- `final_event` is 1 or 0;
- `post_id` is still the one the row started with.

It **writes nothing**. It prints any problems, how many rows you marked reviewed, and a table of the rows where your answer differs from Claude's — read it to confirm those are the changes you meant.

**If problems are listed:** fix those rows in Excel, save, and run this cell again.

In [6]:
# Open labels_review.xlsx, edit the blue columns, set reviewed = Y on rows you checked,
# save, then run this cell. Nothing is written until every row passes.
CODES = {"AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC", "FL", "GA", "HI", "ID", "IL",
         "IN", "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE",
         "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD",
         "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"}

def codes(cell):
    """'ks, tx' -> ['KS', 'TX']. Accepts commas, semicolons or spaces."""
    return sorted({c.strip().upper() for c in re.split(r"[,;\s]+", str(cell)) if c.strip()})

back = pd.read_excel(FOLDER / "labels_review.xlsx", sheet_name="labels", dtype=str).fillna("")
back["row"] = back["row"].astype(float).astype(int).astype(str)
back["final_main"] = back.final_main.str.strip().str.upper()
back["final_event"] = back.final_event.str.strip().str.replace(".0", "", regex=False)

problems = []
# post_id must still be the post it was: compare with labels.csv row by row
original_id = labels.set_index("row").post_id
for r in back.itertuples():
    if r.post_id.strip() != original_id[r.row]:
        problems.append(f"row {r.row}: post_id {r.post_id} does not match labels.csv ({original_id[r.row]})")
    states = codes(r.final_states)
    wrong = [c for c in states + ([r.final_main] if r.final_main else []) if c not in CODES]
    if wrong:
        problems.append(f"row {r.row}: not a state code: {', '.join(wrong)}")
    if r.final_main and r.final_main not in states:
        problems.append(f"row {r.row}: final_main {r.final_main} is not in final_states")
    if states and not r.final_main:
        problems.append(f"row {r.row}: final_states given but no final_main")
    if r.final_event not in ("1", "0"):
        problems.append(f"row {r.row}: final_event must be 1 or 0")

changed = back[(back.final_states.map(codes) != back.claude_states.map(codes))
               | (back.final_main != back.claude_main) | (back.final_event != back.claude_event)]
print("problems:", len(problems)); print("\n".join(problems))
print("rows marked reviewed:", (back.reviewed.str.strip().str.upper() == "Y").sum(),
      "| rows you changed:", len(changed))
changed[["row", "claude_states", "final_states", "claude_main", "final_main",
         "claude_event", "final_event", "comment"]]

problems: 0

rows marked reviewed: 0 | rows you changed: 0


,row,claude_states,final_states,claude_main,final_main,claude_event,final_event,comment


## 7. Save the reviewed labels

Only runs if cell 6 found no problems (otherwise it stops with a message).

1. Copies the current `labels.csv` to `data/llm_pilot/backups/`. It goes in a subfolder because the scoring code reads every CSV in `data/llm_pilot` as a set of results.
2. Writes your answers into `labels.csv`. The post text and IDs come from the original file, never from the spreadsheet.
3. Marks each row you set to `Y` as reviewed by you, with today's date, and adds your comment to that row's note.

In [7]:
# Only runs if cell 6 found no problems. Text and post IDs come from the original
# labels.csv, never from the sheet.
assert not problems, "fix the rows listed in cell 6 first"

backup = FOLDER / "backups" / f"labels.before_review_{date.today()}.csv"
backup.parent.mkdir(exist_ok=True)       # a subfolder: the scoring script reads every CSV in FOLDER
if not backup.exists():
    shutil.copy(FOLDER / "labels.csv", backup)

b = back.set_index("row")
updated = labels.copy()
updated["human_states"] = updated.row.map(lambda r: ",".join(codes(b.at[r, "final_states"])))
updated["human_main"] = updated.row.map(b.final_main)
updated["human_event"] = updated.row.map(b.final_event)
was_reviewed = updated.row.map(b.reviewed.str.strip().str.upper() == "Y")
updated.loc[was_reviewed, "labelled_by"] = f"claude-opus-5-5, reviewed by Max {date.today()}"
comments = updated.row.map(b.comment.str.strip())
updated.loc[comments != "", "note"] = updated.note + " | Max: " + comments

updated.to_csv(FOLDER / "labels.csv", index=False)
print(f"wrote labels.csv ({was_reviewed.sum()} rows reviewed); backup in backups/{backup.name}")

wrote labels.csv (0 rows reviewed); backup in backups/labels.before_review_2026-11-04.csv


## 8. Re-score every method

Scores each method in `data/llm_pilot` against the reviewed labels, first on all 200 posts, then on the high-confidence labels only.

What the columns mean:

- **main state** — share of posts whose main state matches the label (a post with no state counts when both are blank).
- **exact list** — share whose full list of states matches exactly.
- **precision** — of the states a method returned, the share that are right.
- **recall** — of the labelled states, the share the method found.
- **event** — agreement on whether a US event is reported (blank for methods that do not answer it).

Gemma's results are on Google Drive, so its row appears only when `gemma3_27b_v2.csv` is in the folder (or when you run cell 9 of the Colab notebook).

In [2]:
import sys
sys.path.insert(0, "..")                 # the repo root, so scripts/ can be imported
from scripts.score_state_methods import score, show

show(score(FOLDER))
print("\nhigh-confidence labels only:")
show(score(FOLDER, high_confidence=True))

method                 lang    n  main state exact list precision  recall   event
keyword_gazetteer      all   200       31.0%      40.0%     50.6%   96.6%     -  
keyword_gazetteer      en    100       31.0%      35.0%     42.9%   94.0%     -  
keyword_gazetteer      es    100       31.0%      45.0%     59.2%   98.9%     -  
gemma3_27b_v2          all   200       71.0%      58.0%     62.2%   96.9%   81.0%
gemma3_27b_v2          en    100       70.0%      55.0%     64.8%   98.7%   80.0%
gemma3_27b_v2          es    100       72.0%      61.0%     60.1%   95.4%   82.0%
qwen3_32b_v3           all   200       48.0%      68.5%     77.4%   96.3%   87.5%
qwen3_32b_v3           en    100       45.0%      64.0%     73.0%   98.0%   87.0%
qwen3_32b_v3           es    100       51.0%      73.0%     81.8%   94.9%   88.0%
stanza_ner             all   200       62.5%      52.5%     65.2%   93.2%     -  
stanza_ner             en    100       65.0%      55.0%     62.7%   92.6%     -  
stanza_ner      

## 9. Optional: join the sheet to the database

Shows that the reviewed sheet links back to the database through `post_id`: it looks up each post in `meta_post_analysis` and adds its date, page and funnel result.

Needs the Cloud SQL proxy running (`cloud-sql-proxy harvard-capstone-499102:us-central1:h5n1-pg --port 5433`) and `.env` in the repo root. Run after cell 6, which defines the code-tidying function used here.

**Check:** `found in meta_post_analysis: 200`.

In [ ]:
# Needs the Cloud SQL proxy running and .env in the repo root (see README).
# post_id is read as text, so it matches the database exactly.
from h5n1.db import get_engine
from sqlalchemy import text

final = pd.read_excel(FOLDER / "labels_review.xlsx", sheet_name="labels", dtype=str).fillna("")
final["final_states"] = final.final_states.map(lambda c: ",".join(codes(c)))   # 'or' -> 'OR', as in cell 7
final["final_main"] = final.final_main.str.strip().str.upper()
db = pd.read_sql(text("""
        SELECT post_id, creation_time, owner_name, geo_tier, state, disposition
        FROM meta_post_analysis WHERE post_id = ANY(:ids)"""),
    get_engine(), params={"ids": final.post_id.tolist()})
joined = final.merge(db, on="post_id", how="left")
print("rows:", len(joined), "| found in meta_post_analysis:", joined.creation_time.notna().sum())
joined[["row", "post_id", "creation_time", "final_states", "final_main", "state", "disposition"]].head()